# Tennis CTR-GCN Training (Kaggle)

Combined pipeline: AimCLR pretrain → CTR-GCN fine-tune → evaluation.

**Setup:** Create a Kaggle Dataset named `tennis-ctrgcn` containing:
```
tennis-ctrgcn/
├── data/
│   ├── train.npz
│   ├── val.npz
│   └── unlabeled.npz
├── ctrgcn/
│   ├── __init__.py
│   ├── graph.py
│   └── model.py
└── aimclr/
    ├── __init__.py
    ├── augmentations.py
    ├── dataset.py
    ├── loss.py
    └── trainer.py
```

**Accelerator:** GPU T4 × 1 (Settings → Accelerator)

## 1 — Setup

In [ ]:
import sys, os
import torch
import numpy as np

# ── Paths ──────────────────────────────────────────────────────────────────
# Kaggle dataset name: tennis-ctrgcn
INPUT_DIR  = '/kaggle/input/tennis-ctrgcn'
OUTPUT_DIR = '/kaggle/working'

# Add code to Python path
sys.path.insert(0, INPUT_DIR)

TRAIN_NPZ     = f'{INPUT_DIR}/data/train.npz'
VAL_NPZ       = f'{INPUT_DIR}/data/val.npz'
UNLABELED_NPZ = f'{INPUT_DIR}/data/unlabeled.npz'

PRETRAIN_PTH = f'{OUTPUT_DIR}/aimclr_pretrain.pth'
BEST_PTH     = f'{OUTPUT_DIR}/ctrgcn_best.pth'

# Verify files exist
for p in [TRAIN_NPZ, VAL_NPZ, UNLABELED_NPZ]:
    assert os.path.exists(p), f'Missing: {p}'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch: {torch.__version__}')
print(f'Device:  {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU:     {torch.cuda.get_device_name(0)}')

# Verify imports
from ctrgcn.model import CTRGCN
from aimclr.augmentations import compose_augmentation
from aimclr.dataset import UnlabeledSkeletonDataset, LabeledSkeletonDataset
from aimclr.loss import InfoNCELoss
from aimclr.trainer import AimCLRTrainer
print('Imports OK ✓')

## 2 — Load Data

In [ ]:
from collections import Counter

train_data = np.load(TRAIN_NPZ, allow_pickle=True)
val_data   = np.load(VAL_NPZ,   allow_pickle=True)
unlabeled  = np.load(UNLABELED_NPZ)

X_train, y_train = train_data['X'], train_data['y']
X_val,   y_val   = val_data['X'],   val_data['y']
X_unlabeled      = unlabeled['X']

CLASS_NAMES = ['forehand', 'backhand', 'serve', 'volley', 'invalid']
N_CLASSES   = len(CLASS_NAMES)

print(f'Train:     {X_train.shape}  labels: {y_train.shape}')
print(f'Val:       {X_val.shape}   labels: {y_val.shape}')
print(f'Unlabeled: {X_unlabeled.shape}')
print()

train_dist = Counter(y_train.tolist())
val_dist   = Counter(y_val.tolist())
print(f'{"Class":<12} {"Train":>6} {"Val":>5}')
for i, name in enumerate(CLASS_NAMES):
    print(f'{name:<12} {train_dist[i]:>6} {val_dist[i]:>5}')

---
# Part A — AimCLR Self-Supervised Pretraining

## 3 — AimCLR Pretrain

In [ ]:
import math
from torch.utils.data import DataLoader

AIMCLR_EPOCHS = 100
AIMCLR_BATCH  = 64

# Skip AimCLR if too few windows
SKIP_AIMCLR = X_unlabeled.shape[0] < 3000
if SKIP_AIMCLR:
    print(f'⚠ Only {X_unlabeled.shape[0]} windows — skipping AimCLR, using random init.')
else:
    unlabeled_ds = UnlabeledSkeletonDataset(UNLABELED_NPZ, augment_fn=compose_augmentation)
    unlabeled_loader = DataLoader(
        unlabeled_ds, batch_size=AIMCLR_BATCH, shuffle=True,
        num_workers=2, drop_last=True, pin_memory=True,
    )

    backbone = CTRGCN(num_class=N_CLASSES, base_channels=64, adaptive=True)
    trainer = AimCLRTrainer(
        backbone=backbone, feature_dim=256, proj_dim=128,
        temperature=0.1, lr=1e-3, weight_decay=1e-4, device=DEVICE,
    )

    baseline = math.log(2 * AIMCLR_BATCH)
    print(f'Random baseline: log(2×{AIMCLR_BATCH}) = {baseline:.3f}')
    print(f'Batches/epoch: {len(unlabeled_loader)}\n')

    aimclr_losses = trainer.train(
        loader=unlabeled_loader,
        epochs=AIMCLR_EPOCHS,
        save_path=PRETRAIN_PTH,
        print_every=10,
    )

    # Quick loss check
    if aimclr_losses[-1] < baseline * 0.6:
        print(f'\n✓ AimCLR converged: {aimclr_losses[-1]:.3f} < {baseline*0.6:.3f}')
    else:
        print(f'\n⚠ AimCLR did not converge well. Phase 5 will still benefit from partial pretraining.')

## 4 — AimCLR Loss Curve

In [ ]:
import matplotlib.pyplot as plt

if not SKIP_AIMCLR:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(range(1, len(aimclr_losses)+1), aimclr_losses, linewidth=2)
    ax.axhline(baseline, color='r', linestyle='--', alpha=0.5, label=f'Baseline ({baseline:.2f})')
    ax.set_xlabel('Epoch'); ax.set_ylabel('InfoNCE Loss')
    ax.set_title('AimCLR Pretraining'); ax.legend(); ax.grid(alpha=0.3)
    plt.savefig(f'{OUTPUT_DIR}/aimclr_loss.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Skipped — no AimCLR pretraining')

---
# Part B — CTR-GCN Supervised Fine-tuning

## 5 — Model & Optimizer Setup

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# ── Focal Loss ─────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight
    def forward(self, logits, targets):
        ce   = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        pt   = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()

# ── Class weights (inverse frequency) ─────────────────────────────────────
counts = torch.tensor([train_dist[i] for i in range(N_CLASSES)], dtype=torch.float)
class_weights = (counts.sum() / (N_CLASSES * counts)).to(DEVICE)
print(f'Class weights: {[f"{w:.2f}" for w in class_weights.cpu()]}')

# ── Model ──────────────────────────────────────────────────────────────────
model = CTRGCN(num_class=N_CLASSES, base_channels=64, adaptive=True).to(DEVICE)

USE_PRETRAIN = os.path.exists(PRETRAIN_PTH)
if USE_PRETRAIN:
    state = torch.load(PRETRAIN_PTH, map_location=DEVICE)
    missing, unexpected = model.load_state_dict(state, strict=False)
    missing_real = [k for k in missing if not k.startswith('fc.')]
    print(f'Loaded AimCLR pretrain.  Missing (non-FC): {missing_real or "none"}')
else:
    print('Random initialization (no pretrain checkpoint).')

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {n_params:,}')

# ── Optimizer ──────────────────────────────────────────────────────────────
EPOCHS  = 80
WARMUP  = 5

optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, nesterov=True, weight_decay=4e-4)

def lr_lambda(epoch):
    if epoch < WARMUP:
        return (epoch + 1) / WARMUP
    progress = (epoch - WARMUP) / (EPOCHS - WARMUP)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
criterion = FocalLoss(gamma=2.0, weight=class_weights)
print(f'SGD lr=0.1 + warmup({WARMUP}) + cosine({EPOCHS} ep)')

## 6 — DataLoaders

In [ ]:
FINETUNE_BATCH = 64

train_ds = LabeledSkeletonDataset(TRAIN_NPZ, augment_fn=compose_augmentation)
val_ds   = LabeledSkeletonDataset(VAL_NPZ,   augment_fn=None)

train_loader = DataLoader(train_ds, batch_size=FINETUNE_BATCH, shuffle=True,
                          num_workers=2, drop_last=False, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=FINETUNE_BATCH, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)} samples, {len(train_loader)} batches')
print(f'Val:   {len(val_ds)} samples,  {len(val_loader)} batches')

## 7 — Training Loop

In [ ]:
from sklearn.metrics import f1_score

history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
best_val_acc = 0.0
patience_counter = 0
PATIENCE = 15

def run_epoch(loader, train=True):
    model.train(train)
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in loader:
            x = x.unsqueeze(-1).to(DEVICE)
            y = y.to(DEVICE)
            logits = model(x)
            loss = criterion(logits, y)
            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            preds = logits.argmax(dim=1)
            correct    += (preds == y).sum().item()
            total      += y.size(0)
            total_loss += loss.item() * y.size(0)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(y.cpu().tolist())

    avg_loss = total_loss / total
    acc = correct / total
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, acc, f1, all_preds, all_labels


print(f'Training {EPOCHS} epochs  (early stop patience={PATIENCE})\n')
print(f'{"Ep":>4}  {"TrLoss":>8}  {"VaLoss":>8}  {"VaAcc":>6}  {"VaF1":>6}  {"LR":>8}')
print('─' * 55)

for epoch in range(1, EPOCHS + 1):
    tr_loss, _, _, _, _ = run_epoch(train_loader, train=True)
    va_loss, va_acc, va_f1, va_preds, va_labels = run_epoch(val_loader, train=False)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['val_acc'].append(va_acc)
    history['val_f1'].append(va_f1)

    if epoch % 5 == 0 or epoch == 1:
        lr = scheduler.get_last_lr()[0]
        print(f'{epoch:>4}  {tr_loss:>8.4f}  {va_loss:>8.4f}  {va_acc:>6.3f}  {va_f1:>6.3f}  {lr:>8.1e}')

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        best_val_f1 = va_f1
        best_epoch = epoch
        patience_counter = 0
        torch.save(model.state_dict(), BEST_PTH)
        print(f'      ↑ best: acc={va_acc:.3f} f1={va_f1:.3f} → saved')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'\nEarly stop at epoch {epoch}')
            break

print(f'\nBest: epoch {best_epoch}  acc={best_val_acc:.4f}  f1={best_val_f1:.4f}')

## 8 — Final Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

model.load_state_dict(torch.load(BEST_PTH, map_location=DEVICE))
_, final_acc, _, final_preds, final_labels = run_epoch(val_loader, train=False)

print(f'=== Final Val Metrics ===')
print(f'Top-1 Accuracy: {final_acc:.4f}  ({final_acc*100:.1f}%)\n')
print(classification_report(final_labels, final_preds, target_names=CLASS_NAMES, zero_division=0))

# FPR
inv_idx  = CLASS_NAMES.index('invalid')
inv_mask = np.array(final_labels) == inv_idx
fpr = (np.array(final_preds)[inv_mask] != inv_idx).sum() / max(1, inv_mask.sum())
print(f'FPR (invalid→stroke): {fpr:.3f}  ({fpr*100:.1f}%)  [target ≤ 10%]')

# Go/No-Go
f1_per_class = f1_score(final_labels, final_preds, average=None, zero_division=0)
min_f1 = min(f1_per_class[:4])
pass_acc = final_acc >= 0.90
pass_f1  = min_f1   >= 0.85
pass_fpr = fpr      <= 0.10

print(f'\nGo/No-Go Gate:')
print(f'  acc ≥ 90%:  {"✓" if pass_acc else "✗"}  ({final_acc*100:.1f}%)')
print(f'  F1 ≥ 0.85:  {"✓" if pass_f1  else "✗"}  ({min_f1:.3f})')
print(f'  FPR ≤ 10%:  {"✓" if pass_fpr else "✗"}  ({fpr*100:.1f}%)')
print(f'\n{"✓ GO → Phase 6" if all([pass_acc, pass_f1, pass_fpr]) else "✗ NO-GO — need more data"}')

## 9 — Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion matrix
cm = confusion_matrix(final_labels, final_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
im = axes[0].imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
axes[0].set_xticks(range(N_CLASSES)); axes[0].set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
axes[0].set_yticks(range(N_CLASSES)); axes[0].set_yticklabels(CLASS_NAMES)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].set_title('Confusion Matrix')
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        axes[0].text(j, i, f'{cm_norm[i,j]:.2f}', ha='center', va='center',
                     color='white' if cm_norm[i,j]>0.5 else 'black', fontsize=9)
plt.colorbar(im, ax=axes[0])

# Loss
ep = range(1, len(history['train_loss'])+1)
axes[1].plot(ep, history['train_loss'], label='Train')
axes[1].plot(ep, history['val_loss'],   label='Val')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss'); axes[1].set_title('Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

# Acc & F1
axes[2].plot(ep, history['val_acc'], label='Acc', color='green')
axes[2].plot(ep, history['val_f1'],  label='F1',  color='orange')
axes[2].axhline(0.90, color='green',  ls='--', alpha=0.4)
axes[2].axhline(0.85, color='orange', ls='--', alpha=0.4)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Score'); axes[2].set_title('Val Acc & F1')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 10 — Save Metadata

In [ ]:
import json

meta = {
    'val_acc':          round(float(final_acc), 4),
    'val_f1_macro':     round(float(best_val_f1), 4),
    'val_f1_per_class': {CLASS_NAMES[i]: round(float(f1_per_class[i]), 4) for i in range(N_CLASSES)},
    'fpr_invalid':      round(float(fpr), 4),
    'epochs_trained':   len(history['train_loss']),
    'best_epoch':       best_epoch,
    'pretrain_used':    USE_PRETRAIN,
    'go_nogo':          'GO' if all([pass_acc, pass_f1, pass_fpr]) else 'NO-GO',
}
with open(f'{OUTPUT_DIR}/ctrgcn_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(json.dumps(meta, indent=2))
print(f'\nOutput files in {OUTPUT_DIR}/:')
for fn in sorted(os.listdir(OUTPUT_DIR)):
    sz = os.path.getsize(f'{OUTPUT_DIR}/{fn}') / 1e6
    print(f'  {fn}  ({sz:.1f} MB)')
print()
print('Download ctrgcn_best.pth from Output tab → use in Phase 6 export.')